# BIOT 6900 · Module 2 · Assignment 2 — Pancreatic Cancer (CPTAC-PDAC)

Independent multi-omics target discovery. Built on the gene-level schema from Part 3
(`gene, log2fc, pval` per layer -> harmonize -> concordance -> multi-evidence score -> rank -> export),
applied to real CPTAC-PDAC data instead of synthetic/Alzheimer's data.

**Data.** CPTAC-PDAC (LinkedOmics), open access, no data-use agreement:
https://www.linkedomics.org/data_download/CPTAC-PDAC/
- RNA: `mRNA_RSEM_UQ_log2_Tumor.cct` / `mRNA_RSEM_UQ_log2_Normal.cct` (140 tumor / 21 normal)
- Protein: `proteomics_gene_level_MD_abundance_tumor.cct` / `..._normal.cct` (140 tumor / 75 normal)
- Mutation: `Mutation_gene_level.cgt` (140 tumor samples, binary mutation status per gene)

**Note on "unmatched" framing.** Unlike the Alzheimer's assignment, these files ARE sample-matched
(same 140 tumors across RNA + protein + mutation). We still reduce to one row per gene -- computing a
tumor-vs-normal differential (log2fc + p-value) per gene -- so the downstream schema matches the
taught pipeline exactly. This is worth stating explicitly in your report's "matched vs. gene-level"
discussion.

In [1]:
import os
import numpy as np
import pandas as pd
from scipy import stats

EQUAL_WEIGHTS = {"transcriptomic": 1/3, "proteomic": 1/3, "genomic": 1/3}
DATA_DIR = "data_pdac" 

## Helper functions (identical to Part 1/3 -- do not need to edit)

In [2]:
def rank_percentile(series: pd.Series) -> pd.Series:
    """Normalize any score to [0, 1] by rank. Robust to outliers and scale differences."""
    return series.rank(method="average", pct=True)


def multi_evidence_score(df: pd.DataFrame, cols, weights) -> pd.Series:
    """Weighted sum of rank-percentile-normalized layer scores."""
    normed = pd.DataFrame({c: rank_percentile(df[c]) for c in cols})
    return sum(weights[c] * normed[c] for c in cols)

## 3.1 -- Load and reduce each layer to one row per gene

Each raw file is a gene x sample matrix. We reduce each to `gene, log2fc, pval` (RNA, protein) or
`gene, neglog10p` (genomics) -- exactly the schema the rubric grades against.

**Check the printed shapes.** If a matrix looks transposed (~140 rows instead of ~20,000+), add
`.T` after that `pd.read_csv(...)` call.

In [3]:
def load_layer(tumor_path, normal_path):
    t = pd.read_csv(tumor_path, sep="\t", index_col=0)
    n = pd.read_csv(normal_path, sep="\t", index_col=0)
    return t, n

rna_t, rna_n = load_layer(f"{DATA_DIR}/mRNA_RSEM_UQ_log2_Tumor.cct",
                           f"{DATA_DIR}/mRNA_RSEM_UQ_log2_Normal.cct")
prot_t, prot_n = load_layer(f"{DATA_DIR}/proteomics_gene_level_MD_abundance_tumor.cct",
                             f"{DATA_DIR}/proteomics_gene_level_MD_abundance_normal.cct")
mut_t = pd.read_csv(f"{DATA_DIR}/Mutation_gene_level.cgt", sep="\t", index_col=0)
mut_t = mut_t.apply(pd.to_numeric, errors="coerce")

print("RNA tumor:", rna_t.shape, " RNA normal:", rna_n.shape)
print("Protein tumor:", prot_t.shape, " Protein normal:", prot_n.shape)
print("Mutation:", mut_t.shape, "(gene x sample, binary)")

RNA tumor: (28057, 140)  RNA normal: (28057, 21)
Protein tumor: (11662, 140)  Protein normal: (11662, 75)
Mutation: (4424, 140) (gene x sample, binary)


In [4]:
def differential_table(tumor_df, normal_df, lfc_col, p_col):
    """Per-gene tumor-vs-normal log2FC + t-test p-value -> tidy gene table."""
    common_genes = tumor_df.index.intersection(normal_df.index)
    t = tumor_df.loc[common_genes]
    n = normal_df.loc[common_genes]
    lfc = t.mean(axis=1) - n.mean(axis=1)
    pval = pd.Series(
        {g: stats.ttest_ind(t.loc[g], n.loc[g], equal_var=False, nan_policy="omit").pvalue
         for g in common_genes},
        name=p_col)
    out = pd.DataFrame({"gene": common_genes, lfc_col: lfc.values, p_col: pval.values})
    return out.dropna()

tx = differential_table(rna_t, rna_n, "log2fc", "pval")
pr = differential_table(prot_t, prot_n, "log2fc", "pval")
print(f"tx (transcriptomics): {tx.shape}")
print(f"pr (proteomics):      {pr.shape}")
tx.head()

C:\Users\USER\AppData\Local\Temp\ipykernel_32960\2907630620.py:8: SmallSampleWarning: After omitting NaNs, one or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  {g: stats.ttest_ind(t.loc[g], n.loc[g], equal_var=False, nan_policy="omit").pvalue


tx (transcriptomics): (26005, 3)
pr (proteomics):      (11630, 3)


,gene,log2fc,pval
0,A1BG,-0.479548,1.250690e-02
1,A1BG-AS1,-0.178861,4.499880e-01
2,A1CF,-1.463168,2.552527e-08
3,A2M,-0.624133,5.580447e-07
4,A2M-AS1,-0.480572,8.412509e-03


In [6]:
# Genomics Layer: mutation frequency -> enrichment p-value per gene (binomial test vs.
# genome-wide background mutation rate), then -log10(p) as association strength.
mut_freq = mut_t.mean(axis=1, skipna=True).dropna()
n_samples = mut_t.shape[1]
background_rate = mut_freq.mean()

def mut_pval(freq, n=n_samples, bg=background_rate):
    k = round(freq * n)
    return stats.binomtest(k, n, bg, alternative="greater").pvalue

gw = pd.DataFrame({
    "gene": mut_freq.index,
    "mut_freq": mut_freq.values,
    "neglog10p": [-np.log10(max(mut_pval(f), 1e-300)) for f in mut_freq.values],
})
print(f"gw (genomics): {gw.shape}")
gw.sort_values("neglog10p", ascending=False).head()

gw (genomics): (0, 3)


,gene,mut_freq,neglog10p


## 3.2 -- Harmonize on gene symbol -> one joined table

In [7]:
tx2 = tx.rename(columns={"log2fc": "rna_lfc", "pval": "rna_p"})
pr2 = pr.rename(columns={"log2fc": "prot_lfc", "pval": "prot_p"})

df = tx2.merge(pr2, on="gene", how="inner").merge(gw, on="gene", how="inner")
print(f"Genes surviving the 3-way join: {len(df)} "
      f"(RNA {len(tx2)}, protein {len(pr2)}, genomic {len(gw)})")
df.head()

Genes surviving the 3-way join: 0 (RNA 26005, protein 11630, genomic 0)


,gene,rna_lfc,rna_p,prot_lfc,prot_p,mut_freq,neglog10p


## 3.3 -- Sign-agreement concordance
Does each gene move the same direction at RNA and protein (tumor vs. normal)?

In [8]:
df["concordant"] = np.sign(df["rna_lfc"]) == np.sign(df["prot_lfc"])
print(f"Sign-concordant genes: {df['concordant'].sum()} / {len(df)}")

Sign-concordant genes: 0 / 0


## 3.4 -- Multi-evidence score

In [9]:
df["transcriptomic"] = df["rna_lfc"].abs()
df["proteomic"] = df["prot_lfc"].abs()
df["genomic"] = df["neglog10p"]
df["score"] = multi_evidence_score(df, ["transcriptomic", "proteomic", "genomic"], EQUAL_WEIGHTS)
print("scored.")

scored.


## 3.5 -- Rank, inspect, export

Known PDAC drivers to check against: **KRAS, TP53, SMAD4, CDKN2A** (the classic four), plus
CA19-9-related genes (e.g. MUC1) and any others worth a second look.

In [10]:
ranked = df.sort_values("score", ascending=False)
top = ranked.head(15)
ranked.to_csv("targets_pdac.csv", index=False)
print(top[["gene", "rna_lfc", "prot_lfc", "neglog10p", "concordant", "score"]]
      .round(3).to_string(index=False))

Empty DataFrame
Columns: [gene, rna_lfc, prot_lfc, neglog10p, concordant, score]
Index: []


## 3.6 -- Interpretation (write this yourself -- goes in your 3-4 page report)

1. **Disease & data choice.** Why PDAC? CPTAC-PDAC via LinkedOmics is open access, no data-use
   agreement. Note this data is sample-matched (140 tumors profiled at all three layers) even
   though we reduced it to gene-level differentials to match the pipeline schema -- state that
   explicitly, since it affects what you can claim (see point 4).
2. **Weighting.** Equal weights were used here. Argue for keeping them, or justify a change
   (e.g., should mutation enrichment be weighted more heavily than expression change, since
   mutations are more likely to be causal rather than downstream?).
3. **Top targets.** Which known PDAC genes did you recover (KRAS, TP53, SMAD4, CDKN2A, MUC1 ...)?
   Any non-obvious hit worth a second look -- check it against Open Targets or recent literature.
4. **Discordant gene.** Pick a `concordant == False` gene in your top hits (strong RNA, flat/
   opposite protein). What biology could explain RNA and protein disagreeing (post-transcriptional
   regulation, protein turnover, degradation)?
5. **Limitation.** Because your underlying data was sample-matched but you integrated at the
   gene level (tumor-vs-normal summary statistics, not per-patient pairing), what can you claim
   about population-level trends -- and what can't you claim about any individual patient or about
   causality?

## Submit
1. `Kernel -> Restart & Run All`
2. Commit this notebook, `targets_pdac.csv`, and a README (name, disease, each dataset + source
   link + access type) to your `biot6900` repo.
3. Push, confirm files on GitHub, post the repo link on Canvas.